In [ ]:
# Offline Qwen2.5-3B-Instruct report -> 12-label extractor (Kaggle T4, internet OFF).
# Rules-compliant: local model only, no hosted API.
# RUN_ALL=True -> label all 4407 -> pseudo_labels_qwen.csv (with periodic checkpoints).
RUN_ALL = True
import os, glob, re, json, time
import numpy as np, pandas as pd, torch
from transformers import AutoModelForCausalLM, AutoTokenizer
print('torch', torch.__version__, '| cuda avail', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'no-gpu')

cfg = sorted(glob.glob('/kaggle/input/**/config.json', recursive=True),
             key=lambda p: ('qwen' not in p.lower(), len(p)))
ROOT = os.path.dirname(cfg[0])
print('MODEL ROOT:', ROOT)
train_csv = sorted(glob.glob('/kaggle/input/**/train.csv', recursive=True))[0]
print('TRAIN CSV:', train_csv)
df = pd.read_csv(train_csv)
print('shape', df.shape)

In [ ]:
# Qwen2.5-3B-Instruct fp16 fully on the T4 (~6GB, no offload) -> fast, no OOM.
tok = AutoTokenizer.from_pretrained(ROOT)
model = AutoModelForCausalLM.from_pretrained(ROOT, dtype=torch.float16).to('cuda')
model.eval()
print('loaded on', next(model.parameters()).device,
      '| GPU mem GB:', round(torch.cuda.memory_allocated()/1e9, 2))

In [ ]:
KEYS = ['ACL','MCL','MedialMeniscus','LateralMeniscus','MedialOA','LateralOA','PFOA',
        'Effusion','Synovitis','Bakers','Contusion','Fracture']
LABELS = ['ACL','MCL','Medial Meniscus','Lateral Meniscus','Medial OA','Lateral OA','PF OA',
          'Effusion','Synovitis',"Baker's",'Contusion','Fracture']
K2L = dict(zip(KEYS, LABELS))

SYS = ('You are a musculoskeletal radiologist labeling knee MRI reports. Reports may be in '
       'any language (Spanish, English, Dutch, German, etc.). Judge each finding from the '
       'report content, handling negations (intact/normal/no/sin/geen mean ABSENT).')
DEFS = ('For each finding output 1 if the report indicates it is PRESENT/abnormal, else 0 '
        '(absent, normal, or not mentioned). Findings:\n'
        '- ACL: anterior cruciate ligament tear/rupture/sprain\n'
        '- MCL: medial collateral ligament tear/injury\n'
        '- MedialMeniscus: medial meniscus tear\n'
        '- LateralMeniscus: lateral meniscus tear\n'
        '- MedialOA: medial compartment osteoarthritis / cartilage loss / osteophytes\n'
        '- LateralOA: lateral compartment osteoarthritis / cartilage loss / osteophytes\n'
        '- PFOA: patellofemoral or trochlear osteoarthritis / chondral loss\n'
        '- Effusion: joint effusion / joint fluid\n'
        '- Synovitis: synovitis / synovial thickening\n'
        '- Bakers: Baker (popliteal) cyst\n'
        '- Contusion: bone contusion / bruise / bone-marrow edema\n'
        '- Fracture: fracture\n'
        'Respond with ONLY a JSON object with exactly these keys and 0/1 values: '
        + ', '.join(KEYS) + '.')

def parse(txt):
    m = re.search(r'\{[^{}]*\}', txt, re.DOTALL)
    d = {}
    if m:
        try:
            d = json.loads(m.group(0))
        except Exception:
            for k in KEYS:
                mm = re.search(rf'"{k}"\s*:\s*(\d)', m.group(0))
                if mm: d[k] = int(mm.group(1))
    return {L: int(d.get(k, 0) or 0) for k, L in K2L.items()}

@torch.no_grad()
def extract(report):
    msgs = [{'role':'system','content':SYS},
            {'role':'user','content':DEFS + '\n\nREPORT:\n' + str(report)[:4000]}]
    text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inp = tok(text, return_tensors='pt').to(model.device)
    out = model.generate(**inp, max_new_tokens=160, do_sample=False,
                         pad_token_id=tok.eos_token_id)
    gen = tok.decode(out[0][inp.input_ids.shape[1]:], skip_special_tokens=True)
    return parse(gen), gen

p, raw = extract(df['Report'].iloc[0])
print('RAW:', raw[:300])
print('PARSED:', p)


In [ ]:
gold = df[df['ACL'].notna()].reset_index(drop=True)
t0 = time.time(); rows = []
for i, r in gold.iterrows():
    p, _ = extract(r['Report']); rows.append(p)
    if i % 10 == 0: print(f'{i}/{len(gold)}  {time.time()-t0:.0f}s', flush=True)
pred = pd.DataFrame(rows)[LABELS]
print(f'gold inference: {time.time()-t0:.0f}s for {len(gold)}')

print(f"\n{'label':18s} {'TP':>3} {'FP':>3} {'FN':>3} {'TN':>3}  {'prec':>5} {'rec':>5} {'AUC':>5}")
aucs = []
for lab in LABELS:
    y = (gold[lab].values == 1.0).astype(int); pp = pred[lab].values.astype(int)
    tp=int(((pp==1)&(y==1)).sum()); fp=int(((pp==1)&(y==0)).sum())
    fn=int(((pp==0)&(y==1)).sum()); tn=int(((pp==0)&(y==0)).sum())
    prec=tp/(tp+fp) if tp+fp else 0.0; rec=tp/(tp+fn) if tp+fn else 0.0
    tpr=rec; tnr=tn/(tn+fp) if tn+fp else 0.0; auc=(tpr+tnr)/2; aucs.append(auc)
    print(f'{lab:18s} {tp:3d} {fp:3d} {fn:3d} {tn:3d}  {prec:5.2f} {rec:5.2f} {auc:5.3f}')
print(f'\nMACRO AUC on gold-58 (Qwen): {np.mean(aucs):.4f}   (rule-v0 = 0.6534, baseline = 0.5)')


In [ ]:
if RUN_ALL:
    LABELS_ = LABELS
    goldmask = df['ACL'].notna().values
    t0 = time.time(); allrows = []
    def flush(n):
        ap = pd.DataFrame(allrows)[LABELS_]
        out = df.iloc[:n][['StudyInstanceUID']].copy()
        gm = goldmask[:n]
        for lab in LABELS_:
            out[lab] = np.where(gm, df.iloc[:n][lab].values, ap[lab].values.astype(float))
        out.to_csv('pseudo_labels_qwen.csv', index=False)  # overwrite = latest checkpoint
    for i, r in df.iterrows():
        p, _ = extract(r['Report']); allrows.append(p)
        if i % 200 == 0:
            print(f'{i}/{len(df)}  {time.time()-t0:.0f}s', flush=True)
            if i and i % 500 == 0: flush(i + 1); print('  checkpoint saved', flush=True)
    flush(len(df))
    print('wrote pseudo_labels_qwen.csv', (len(df), len(LABELS_)+1), f'{time.time()-t0:.0f}s')
else:
    print('RUN_ALL=False -> validation only')